<a href="https://colab.research.google.com/github/Eklavya112008/abtalks/blob/main/day16.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import json
import random

# -----------------------------
# Simulated RAG pipeline
# -----------------------------

class RAGPipeline:
    def __init__(self, corpus):
        self.corpus = corpus
        self.results_file = "rag_results.json"

    def embed(self, text):
        """Simulate embeddings by hashing words into a pseudo-vector."""
        return sum(ord(c) for c in text) % 1000

    def similarity(self, q_emb, d_emb):
        """Simple similarity score."""
        return 1 - abs(q_emb - d_emb) / 1000

    def retrieve(self, query, top_k=2):
        q_emb = self.embed(query)
        scored = []
        for doc in self.corpus:
            d_emb = self.embed(doc)
            score = self.similarity(q_emb, d_emb)
            scored.append((doc, score))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]

    def generate(self, query, chunks):
        """Simulate generation with possible errors."""
        # If no chunks, hallucinate
        if not chunks:
            return "No relevant information found."
        # Randomly introduce wrong answers
        if "Inception" in query:
            return "Interstellar was directed by Christopher Nolan"
        if "Hamlet" in query:
            return "Christopher Marlowe wrote Hamlet"
        if "World War II" in query:
            return "World War II ended in 1946"
        if "square root of 144" in query:
            return "The square root of 144 is 10"
        # Otherwise echo top chunk
        return chunks[0][0]

    def run(self, query, failure_type, explanation):
        chunks = self.retrieve(query)
        answer = self.generate(query, chunks)
        log_entry = {
            "query": query,
            "retrieved_chunks": [{"content": c[0], "score": round(c[1], 2)} for c in chunks],
            "final_answer": answer,
            "failure_type": failure_type,
            "explanation": explanation
        }
        self.log_result(log_entry)
        return log_entry

    def log_result(self, entry):
        try:
            with open(self.results_file, "r") as f:
                data = json.load(f)
        except FileNotFoundError:
            data = []
        data.append(entry)
        with open(self.results_file, "w") as f:
            json.dump(data, f, indent=2)

# -----------------------------
# Corpus (toy examples)
# -----------------------------
corpus = [
    "Water boils at 100 degrees Celsius.",
    "Water freezes at 0 degrees Celsius.",
    "Interstellar was directed by Christopher Nolan.",
    "Inception is a 2010 film directed by Christopher Nolan.",
    "The capital of Austria is Vienna.",
    "The capital of Australia is Canberra.",
    "World War II ended in 1945.",
    "Hamlet was written by William Shakespeare.",
    "The square root of 144 is 12.",
    "Apple is a technology company.",
    "Apple is a fruit.",
    "Java is a programming language.",
    "Java is an island in Indonesia.",
    "Mercury is a planet.",
    "Mercury is a chemical element.",
    "Mercury is a Roman god."
]

pipeline = RAGPipeline(corpus)

# -----------------------------
# Test Suite (15 queries)
# -----------------------------
queries = [
    ("Who won the Nobel Prize in Physics in 2023?", "Retrieval Failure", "Corpus lacked Nobel 2023 data."),
    ("Explain the capital of Gondor.", "Retrieval Failure", "Fictional entity, no chunk retrieved."),
    ("What is the latest Mars rover mission update?", "Retrieval Failure", "Outdated corpus, no recent Mars rover info."),
    ("Summarize the entire Harry Potter series in detail.", "Context Window Overflow", "Too many chunks, truncation lost key info."),
    ("Give me all laws passed in India in 2022.", "Context Window Overflow", "Legal corpus exceeded window size."),
    ("Explain every theorem in linear algebra.", "Context Window Overflow", "Linear algebra theorems overflowed context."),
    ("What is the boiling point of water?", "Answer-Context Mismatch", "Freezing point chunk retrieved."),
    ("Who directed Inception?", "Answer-Context Mismatch", "Chunk about Interstellar confused with Inception."),
    ("What is the capital of Australia?", "Answer-Context Mismatch", "Austria chunk confused with Australia."),
    ("Tell me about Apple.", "Vague Context", "Fruit vs company ambiguity."),
    ("Explain Java.", "Vague Context", "Island vs programming language ambiguity."),
    ("What is Mercury?", "Vague Context", "Planet vs element vs god ambiguity."),
    ("What year did World War II end?", "Correct Chunk, Wrong Answer", "Chunk said 1945, model hallucinated 1946."),
    ("Who wrote Hamlet?", "Correct Chunk, Wrong Answer", "Chunk said Shakespeare, model hallucinated Marlowe."),
    ("What is the square root of 144?", "Correct Chunk, Wrong Answer", "Chunk said 12, model hallucinated 10.")
]

results = []
for q, ftype, expl in queries:
    results.append(pipeline.run(q, ftype, expl))

# -----------------------------
# Scorecard
# -----------------------------
scorecard = []
for r in results:
    # crude scoring
    retrieval_score = 1 if r["failure_type"] == "Retrieval Failure" else 3 if r["failure_type"] in ["Answer-Context Mismatch","Vague Context"] else 4
    answer_score = 1 if r["failure_type"] in ["Retrieval Failure","Answer-Context Mismatch"] else 2
    scorecard.append({"query": r["query"], "retrieval": retrieval_score, "answer": answer_score})

avg_retrieval = sum(s["retrieval"] for s in scorecard)/len(scorecard)
avg_answer = sum(s["answer"] for s in scorecard)/len(scorecard)

print("Scorecard:")
for s in scorecard:
    print(f"- {s['query']}\n  Retrieval: {s['retrieval']}, Answer: {s['answer']}")
print(f"\nAverage Retrieval Quality: {avg_retrieval:.2f}")
print(f"Average Answer Quality: {avg_answer:.2f}")


Scorecard:
- Who won the Nobel Prize in Physics in 2023?
  Retrieval: 1, Answer: 1
- Explain the capital of Gondor.
  Retrieval: 1, Answer: 1
- What is the latest Mars rover mission update?
  Retrieval: 1, Answer: 1
- Summarize the entire Harry Potter series in detail.
  Retrieval: 4, Answer: 2
- Give me all laws passed in India in 2022.
  Retrieval: 4, Answer: 2
- Explain every theorem in linear algebra.
  Retrieval: 4, Answer: 2
- What is the boiling point of water?
  Retrieval: 3, Answer: 1
- Who directed Inception?
  Retrieval: 3, Answer: 1
- What is the capital of Australia?
  Retrieval: 3, Answer: 1
- Tell me about Apple.
  Retrieval: 3, Answer: 2
- Explain Java.
  Retrieval: 3, Answer: 2
- What is Mercury?
  Retrieval: 3, Answer: 2
- What year did World War II end?
  Retrieval: 4, Answer: 2
- Who wrote Hamlet?
  Retrieval: 4, Answer: 2
- What is the square root of 144?
  Retrieval: 4, Answer: 2

Average Retrieval Quality: 3.00
Average Answer Quality: 1.60
